In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings
sales= pd.read_csv("sales_date.csv")
sales.dropna(inplace= True)
sales.isnull().sum()

In [ ]:
sales.shape

In [ ]:
sales.groupby('customer_id').size().sort_values(ascending=False).reset_index(name='order_count').head()

# Feature Creation Cus*order_date ---------- Product ------  

In [ ]:
df = sales#.sample(100)#[sales["customer_id"] == 24271].copy()
df["order_date"] = pd.to_datetime(df["order_date"])
df["day_of_week"] = df["order_date"].dt.dayofweek
df["day_of_month"] = df["order_date"].dt.day
df["month"] = df["order_date"].dt.month
df["quarter"] = df["order_date"].dt.quarter
df["week_of_year"] = df["order_date"].dt.isocalendar().week
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

In [ ]:
df['spend'] =df['price_after_discount']*df['product_count']
df.head()

In [ ]:
df.shape

In [ ]:
import pandas as pd
import numpy as np
# Keep only required columns
cols = ["customer_id", "order_date", "order_id", "product_id",
    "product_count", "spend", "price_after_discount",
    "product_discount", "product_price"]
df2 = df[cols].copy()
group_cols = ["customer_id", "order_date"]
# -----------------------------------
# 1. Create groupby ONCE
# -----------------------------------
g = df2.groupby(group_cols, sort=False, observed=True)
# -----------------------------------
# 2. Basic aggregations
# -----------------------------------
cpd = g.agg(
    purchase_count=("order_id", "count"),
# Quantity
    avg_quantity=("product_count", "mean"),
    max_quantity=("product_count", "max"),
# Spend
    total_spend=("spend", "sum"),
    avg_spend=("spend", "mean"),
    min_spend=("spend", "min"),
    max_spend=("spend", "max"),
# Price after discount
    total_pad=("price_after_discount", "sum"),
    avg_pad=("price_after_discount", "mean"),
    min_pad=("price_after_discount", "min"),
    max_pad=("price_after_discount", "max"),
# Product count
    total_pc=("product_count", "sum"),
    avg_pc=("product_count", "mean"),
    min_pc=("product_count", "min"),
    max_pc=("product_count", "max"),
# Product discount
    total_pd=("product_discount", "sum"),
    avg_pd=("product_discount", "mean"),
    min_pd=("product_discount", "min"),
    max_pd=("product_discount", "max"),
 # Product price
    total_pp=("product_price", "sum"),
    avg_pp=("product_price", "mean"),
    min_pp=("product_price", "min"),
    max_pp=("product_price", "max"),)
# -----------------------------------
# 3. Calculate ALL quantiles together
# -----------------------------------
quantile_cols = [ "product_count", "spend","price_after_discount", "product_discount","product_price"]
qs = [0.25, 0.50, 0.75, 0.90, 0.95]
quantiles = (g[quantile_cols] .quantile(qs)  .unstack(level=-1))
# -----------------------------------
# 4. Rename quantile columns
# -----------------------------------
prefix = {"product_count": "pc","spend": "spend","price_after_discount": "pad","product_discount": "pd","product_price": "pp"}
quantiles.columns = [ f"p{int(q * 100)}_{prefix[col]}" for col, q in quantiles.columns]
# Quantity names from your original code
quantiles["p25_quantity"] = quantiles["p25_pc"]
quantiles["p50_quantity"] = quantiles["p50_pc"]
quantiles["p75_quantity"] = quantiles["p75_pc"]
quantiles["p90_quantity"] = quantiles["p90_pc"]
quantiles["p95_quantity"] = quantiles["p95_pc"]
# -----------------------------------
# 5. Mode product
# -----------------------------------
mode_product = (df2.groupby(group_cols + ["product_id"], sort=False).size().rename("count").reset_index().sort_values("count", ascending=False)
    .drop_duplicates(group_cols).set_index(group_cols)["product_id"].rename("mode_product_id"))
# -----------------------------------
# 6. Combine everything
# -----------------------------------
cpd = (cpd.join(quantiles).join(mode_product).reset_index())
print(cpd.shape)
print(cpd.head())

In [ ]:
columns = cpd.columns[3:]

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 100)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

In [ ]:
cpd = cpd.sort_values( ["customer_id", "order_date"]).copy()
for i in cpd.columns:
    cpd[f"last_{i}"] = (cpd.groupby("customer_id")[i] .shift(1) )

In [ ]:
columns = cpd.columns[3:]

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 100)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

In [ ]:
cpd.head()

In [ ]:
cpd.columns

In [ ]:
cpd["Same_order"] = (cpd["mode_product_id"] == cpd["last_mode_product_id"]).astype(int)
cpd["Last_transaction_days"] = (cpd["order_date"] - cpd["last_order_date"] ).dt.days
cpd.drop(['last_customer_id','last_mode_product_id'],axis=1 , inplace = True)

In [ ]:
cpd.columns

In [ ]:
new_cols = [    'purchase_count', 'avg_quantity', 'max_quantity',
                #'p25_quantity','p50_quantity', 'p75_quantity', 'p90_quantity', 'p95_quantity',
                'total_spend', 'avg_spend', 'min_spend', 'max_spend',
                #'p25_spend','p50_spend', 'p75_spend', 'p90_spend', 'p95_spend',
                'total_pad','avg_pad', 'min_pad', 'max_pad', 
                #'p25_pad', 'p50_pad', 'p75_pad','p90_pad', 'p95_pad',
                'total_pc', 'avg_pc', 'min_pc', 'max_pc',
                #'p25_pc', 'p50_pc', 'p75_pc', 'p90_pc', 'p95_pc', 
                'total_pd', 'avg_pd','min_pd', 'max_pd', 
                #'p25_pd', 'p50_pd', 'p75_pd', 'p90_pd', 'p95_pd',
                'total_pp', 'avg_pp', 'min_pp', 'max_pp',
                #'p25_pp', 'p50_pp', 'p75_pp','p90_pp', 'p95_pp' 
           ]

In [ ]:
import pandas as pd
import numpy as np
import time
start = time.time()
# --------------------------------------------------
# 1. Prepare data ONCE
# --------------------------------------------------
group_cols = ["customer_id"]
windows = [2, 3, 7, 30]
cpd["order_date"] = pd.to_datetime(cpd["order_date"])
cpd = (cpd.sort_values(["customer_id", "order_date"]).reset_index(drop=True))
# --------------------------------------------------
# 2. Process each feature
# --------------------------------------------------
for col in new_cols:
# Historical value only — prevents data leakage
    hist = cpd.groupby("customer_id", sort=False)[col].shift(1)
 # Temporary dataframe
    temp = pd.DataFrame({"customer_id": cpd["customer_id"].values,"order_date": cpd["order_date"].values, "value": hist.values})
# --------------------------------------------------
    # 3. Calculate rolling statistics
    # --------------------------------------------------
for days in windows:
r = (temp.set_index("order_date").groupby("customer_id", sort=False)["value"].rolling(f"{days}D",min_periods=1,closed="both") )
 # Basic statistics
        cpd[f"{col}_{days}d_mean"] = r.mean().to_numpy()
        cpd[f"{col}_{days}d_min"]  = r.min().to_numpy()
        cpd[f"{col}_{days}d_max"]  = r.max().to_numpy()
        cpd[f"{col}_{days}d_std"]  = r.std().to_numpy()
 # Quantiles
        for q, name in [(0.25, "p25"),(0.50, "p50"),(0.75, "p75"),(0.90, "p90"),(0.95, "p95")]:
            cpd[f"{col}_{days}d_{name}"] = (
                r.quantile(q).to_numpy())
            del r
del temp, hist
print(f"Completed: {col} |"f"Time: {(time.time() - start):.1f}s")
# --------------------------------------------------
# 4. Final sorting
# --------------------------------------------------
cpd = (cpd.sort_values(["customer_id", "order_date"]).reset_index(drop=True))
print("\nFinished!")
print(f"Total time: {time.time() - start:.2f} seconds")
print("Final shape:", cpd.shape)

In [ ]:
columns = cpd.columns[3:]

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 100)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

In [ ]:
cpd.columns

In [ ]:
columns = cpd.columns

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 10)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

In [ ]:
cpd.rename({'mode_product_id':'product_id'},axis=1 , inplace=True)

In [ ]:
columns = cpd.columns

print("=" * 100)
print(f"Total columns: {(cpd.shape)}")
print("=" * 10)
for i in range(0, len(columns), 10):
    print(columns[i:i+10].tolist())

In [ ]:
cpd.to_csv("Final_Feature_code.csv")

In [6]:
import pandas as pd
import numpy as np
sales= pd.read_csv("Final_Feature_code.csv")
sales.drop('order_date',axis=1, inplace= True)
sales.drop('customer_id',axis=1, inplace= True)
sales.drop('Unnamed: 0',axis=1, inplace= True)
sales.columns


Index(['purchase_count', 'avg_quantity', 'max_quantity', 'total_spend',
       'avg_spend', 'min_spend', 'max_spend', 'total_pad', 'avg_pad',
       'min_pad',
       ...
       'last_p75_pp', 'last_p90_pp', 'last_p95_pp', 'last_p25_quantity',
       'last_p50_quantity', 'last_p75_quantity', 'last_p90_quantity',
       'last_p95_quantity', 'Same_order', 'Last_transaction_days'],
      dtype='object', length=110)

In [7]:
sales= sales.drop('last_order_date',axis=1)

In [8]:
X_train= sales.drop('product_id',axis=1)
y_train = sales['product_id'] 

In [11]:
import numpy as np
import pandas as pd
import gc
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif

# ============================================================
# 0. HANDLE NaNs FIRST (this is what crashed your code)
# ============================================================
print("NaN counts per column:\n", X_train.isna().sum()[X_train.isna().sum() > 0])


NaN counts per column:
 Series([], dtype: int64)


In [12]:

# Simple, fast fix: fill numeric NaNs with median, categorical with mode
for col in X_train.columns:
    if X_train[col].isna().sum() > 0:
        if X_train[col].dtype in ['float64', 'int64']:
            X_train[col] = X_train[col].fillna(X_train[col].median())
        else:
            X_train[col] = X_train[col].fillna(X_train[col].mode()[0])

# ============================================================
# 1. HANDLE CATEGORICAL COLUMNS (RF/MI need numeric input)
# ============================================================
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
print("Categorical columns:", cat_cols)

for col in cat_cols:
    X_train[col] = X_train[col].astype('category').cat.codes

# ============================================================
# 2. SAMPLE DATA FOR TRAINING (smaller = much faster)
# ============================================================
MODEL_SIZE = 100000   # reduced from 300k -> big speed win, minimal accuracy loss
MI_SIZE = 2000        # MI is slow; keep this small

if len(X_train) > MODEL_SIZE:
    X_model = X_train.sample(n=MODEL_SIZE, random_state=42)
    y_model = y_train.loc[X_model.index]
else:
    X_model = X_train
    y_model = y_train

print("Total training rows :", len(X_train))
print("Rows used for model  :", len(X_model))


Categorical columns: []
Total training rows : 1410027
Rows used for model  : 100000


In [13]:

# ============================================================
# 3. TRAIN RANDOM FOREST (optimized for speed)
# ============================================================
model = RandomForestClassifier(
    n_estimators=100,       # reduced from 200 -> ~2x faster, similar ranking
    max_depth=8,
    max_features='sqrt',   # big speed boost, standard for RF
    min_samples_leaf=5,    # reduces overfitting + speeds up splits
    random_state=42,
    n_jobs=-1
)
model.fit(X_model, y_model)
print("RandomForest training completed!")


RandomForest training completed!


In [14]:

# ============================================================
# 4. FEATURE IMPORTANCE
# ============================================================
importance = model.feature_importances_
rf_df = pd.DataFrame({
    "feature": X_train.columns,
    "rf_importance": importance
})
rf_df["rf_pct"] = rf_df["rf_importance"] / rf_df["rf_importance"].sum() * 100
rf_df = rf_df.sort_values("rf_importance", ascending=False).reset_index(drop=True)
rf_df["rf_rank"] = rf_df.index + 1


In [15]:

# ============================================================
# 5. MUTUAL INFORMATION (small sample + fewer neighbors = faster)
# ============================================================
X_mi = X_model.sample(n=min(MI_SIZE, len(X_model)), random_state=42)
y_mi = y_model.loc[X_mi.index]

mi_scores = mutual_info_classif(
    X_mi, y_mi,
    discrete_features='auto',
    n_neighbors=3,      # default is fine, lower = faster, less precise
    random_state=42,
    n_jobs=-1           # parallelize
)

mi_df = pd.DataFrame({
    "feature": X_train.columns,
    "mi_score": mi_scores
})
mi_df["mi_pct"] = mi_df["mi_score"] / mi_df["mi_score"].sum() * 100
mi_df = mi_df.sort_values("mi_score", ascending=False).reset_index(drop=True)
mi_df["mi_rank"] = mi_df.index + 1

# ============================================================
# 6. COMBINE BOTH METHODS
# ============================================================
combined = rf_df.merge(mi_df, on="feature")
combined["avg_rank"] = (combined["rf_rank"] + combined["mi_rank"]) / 2
combined = combined.sort_values("avg_rank").reset_index(drop=True)

print("\n" + "=" * 70)
print("COMBINED FEATURE IMPORTANCE (RandomForest + Mutual Information)")
print("=" * 70)
print(combined[["feature", "rf_pct", "mi_pct", "rf_rank", "mi_rank", "avg_rank"]].to_string(index=False))



COMBINED FEATURE IMPORTANCE (RandomForest + Mutual Information)
              feature   rf_pct   mi_pct  rf_rank  mi_rank  avg_rank
               min_pp 5.022545 4.094213        2        1       1.5
               p90_pp 5.137934 3.987048        1        7       4.0
              p95_pad 4.849004 4.001216        4        6       5.0
               p25_pp 5.004503 3.983084        3        8       5.5
               p95_pp 4.689673 4.003434        6        5       5.5
               max_pp 4.433072 4.053039       10        3       6.5
              min_pad 4.415078 4.089341       11        2       6.5
               p75_pp 4.835394 3.967906        5       11       8.0
              max_pad 3.961733 4.039395       13        4       8.5
              p90_pad 4.541855 3.982156        8        9       8.5
               avg_pp 4.576000 3.963679        7       12       9.5
               p50_pp 4.533751 3.963288        9       13      11.0
              p25_pad 3.917628 3.970821       14   

In [17]:
#=============================================================
# 7. SELECT TOP FEATURES
# ============================================================
TOP_N = 20
selected_features = combined.head(TOP_N)["feature"].tolist()
print("\nSelected Top", TOP_N, "Features:")
print(selected_features)

# ============================================================
# 8. CLEAN MEMORY
# ============================================================
del X_model, X_mi
gc.collect()


Selected Top 20 Features:
['min_pp', 'p90_pp', 'p95_pad', 'p25_pp', 'p95_pp', 'max_pp', 'min_pad', 'p75_pp', 'max_pad', 'p90_pad', 'avg_pp', 'p50_pp', 'p25_pad', 'p50_pad', 'avg_pad', 'p75_pad', 'p90_pd', 'total_pad', 'max_pd', 'p95_pd']


48

In [18]:
selected_features = [
    'min_pp', 'p90_pp', 'p95_pad', 'p25_pp', 'p95_pp', 'max_pp', 'min_pad',
    'p75_pp', 'max_pad', 'p90_pad', 'avg_pp', 'p50_pp', 'p25_pad', 'p50_pad',
    'avg_pad', 'p75_pad', 'p90_pd', 'total_pad', 'max_pd', 'p95_pd',
    'total_pp', 'min_pd', 'p50_pd', 'p75_pd', 'p25_pd', 'avg_pd', 'total_pd'
]

print("Selected features:", len(selected_features))
X_train_selected = X_train[selected_features]

Selected features: 27


In [19]:
features_to_delete = [
    'avg_spend', 'min_spend', 'p25_spend', 'p95_spend', 'total_spend', 'max_spend',
    'p50_spend', 'p90_spend', 'p75_spend', 'purchase_count', 'avg_quantity', 'p50_pc',
    'total_pc', 'max_pc', 'last_p25_pp', 'Last_transaction_days', 'last_max_pd',
    'avg_pc', 'p25_pc', 'max_quantity', 'p95_pc', 'last_p90_pd', 'last_p95_pp',
    'last_max_spend', 'p50_quantity', 'p25_quantity', 'p90_quantity', 'last_min_pp',
    'last_total_pp', 'last_total_spend', 'last_p95_pd', 'last_avg_pd', 'last_max_pp',
    'last_total_pad', 'last_avg_spend', 'last_min_pad', 'last_p75_pd', 'last_p25_pc',
    'last_avg_pad', 'last_p50_spend', 'p75_pc', 'last_avg_pp', 'last_p95_spend',
    'last_min_spend', 'last_p25_pd', 'last_p75_pp', 'last_p90_quantity', 'last_p75_spend',
    'last_min_pc', 'last_purchase_count', 'last_max_pad', 'last_p25_spend', 'last_min_pd',
    'last_p95_pad', 'min_pc', 'p90_pc', 'last_p50_quantity', 'last_p90_pc', 'last_p50_pc',
    'last_max_pc', 'last_avg_pc', 'last_avg_quantity', 'last_p25_pad', 'last_max_quantity',
    'p75_quantity', 'last_total_pc', 'last_p90_spend', 'last_p75_pc', 'p95_quantity',
    'last_p75_pad', 'last_total_pd', 'last_p95_pc', 'last_p50_pp', 'last_p25_quantity',
    'last_p90_pp', 'last_p50_pad', 'last_p95_quantity', 'last_p50_pd', 'last_p75_quantity',
    'last_p90_pad', 'Same_order'
]

print("Features to delete:", len(features_to_delete))
X_train_reduced = X_train.drop(columns=features_to_delete)
print("Final shape:", X_train_reduced.shape)

Features to delete: 81
Final shape: (1410027, 27)


In [20]:
# 1. Create a new virtual environment
python -m venv myenv

# 2. Activate it
# Windows:
myenv\Scripts\activate
# Mac/Linux:
source myenv/bin/activate

# 3. Install packages INTO this environment
pip install ipykernel lightgbm shap scikit-learn pandas numpy jupyter

# 4. Register this environment as a Jupyter kernel
python -m ipykernel install --user --name=myenv --display-name="New Python "

SyntaxError: invalid syntax (1213779720.py, line 2)